# Predicting Global Food Loss and Waste

Orchestration notebook. All modelling code lives in `src/`; this notebook
loads it, runs the experiments, and displays the results.

| Module | Responsibility |
|---|---|
| `data_prep` | loading, filtering, feature and target definitions |
| `pipelines` | all eight model configurations (single source of truth) |
| `baselines` | persistence and linear-trend forecasts |
| `validation` | walk-forward time-aware validation |
| `ablations` | year-treatment ablation, hyperparameter searches |
| `visualization` | prediction helpers and plots |

In [ ]:
import sys, pathlib, warnings

REPO_ROOT = pathlib.Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt

from data_prep import load_flw_data, describe_dataset, DENSE_YEAR_MAX, GROUP_COLS, TARGET
from pipelines import MODEL_NAMES, TUNED_PARAMS, build_pipeline
from ablations import year_ablation, save_table
from validation import walk_forward, summarise, save_results
from visualization import (
    predict_food_loss,
    visualize_food_loss_by_stage,
    visualize_food_loss_comparison,
    plot_raw_data_overview,
)

pd.set_option("display.width", 140)
print("modules loaded")

## 1. Dataset

In [ ]:
flw_data = load_flw_data()
pd.Series(describe_dataset(flw_data)).to_frame("value")

In [ ]:
_ = plot_raw_data_overview(flw_data)
plt.show()

## 2. Does the year feature earn its place?

Each model is fitted three ways on the same random 80/20 split: with `year`
dropped entirely, passed through unscaled, and standardised.

In [ ]:
ablation = year_ablation(flw_data)
save_table(ablation, "year_ablation.csv")
ablation.pivot(index="model", columns="year_mode", values="test_r2").round(4)

## 3. Time-aware validation

A random split lets a model see the same country/commodity/stage in both
halves. Training on years <= Y and testing on years > Y instead measures
whether the models forecast at all. Repeating over several cutoffs gives a
spread rather than a single number, and both naive baselines are scored on
the identical splits.

In [ ]:
validation_data = load_flw_data(year_max=DENSE_YEAR_MAX)
results, predictions = walk_forward(validation_data)
summary = summarise(results)
save_results(results, predictions, summary)
summary

## 4. Predictive capabilities

Two representative models: the neural network standing in for those fitted
without a temporal feature, and the decision tree for those fitted with
one.

In [ ]:
X_year, y = flw_data[GROUP_COLS + ["year"]], flw_data[TARGET]
X_no_year = flw_data[GROUP_COLS]

neural_pipeline = build_pipeline("Neural Network", year_mode="none")
neural_pipeline.fit(X_no_year, y)

dtpipeline = build_pipeline("Decision Tree", year_mode="raw")
dtpipeline.fit(X_year, y)

print("fitted:", type(neural_pipeline[-1]).__name__, "and", type(dtpipeline[-1]).__name__)

In [ ]:
predict_food_loss(neural_pipeline, "Rice", "China", "Storage")
predict_food_loss(dtpipeline, "Rice", "Benin", "Storage", year=2024)

### Loss by supply-chain stage

In [ ]:
_ = visualize_food_loss_by_stage("Rice", "China", neural_pipeline, flw_data)
plt.show()

In [ ]:
_ = visualize_food_loss_by_stage("Rice", "China", dtpipeline, flw_data, year=2021)
plt.show()

A year with no observations shows predictions alone rather than failing.

In [ ]:
_ = visualize_food_loss_by_stage("Rice", "China", dtpipeline, flw_data, year=2030)
plt.show()

### Loss by commodity

In [ ]:
_ = visualize_food_loss_comparison("Benin", "Storage", neural_pipeline, flw_data)
plt.show()

In [ ]:
_ = visualize_food_loss_comparison("Benin", "Storage", dtpipeline, flw_data, year=2030)
plt.show()